In [0]:
%pip install pytest

In [0]:
# ============================================================
# CELL 1: INITIALIZE HUMAN GATEWAY
# ============================================================

from Human_GateWay.human_gateway import HumanGateway
import json

REVIEW_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/human_reviews"
)

gateway = HumanGateway(review_dir=REVIEW_DIR)

print("=" * 80)
print("HUMAN GATEWAY INITIALIZED")
print("=" * 80)

print("Review directory:", REVIEW_DIR)
print("Gateway ready!")

In [0]:
# ============================================================
# CELL 2: VERIFY EXISTING APPROVALS
# ============================================================

APPROVED_REVIEWS = {
    "m_customer_standardize":
        "f5f2499d-d3c9-4475-bed5-3cf7cf2085ff",

    "m_order_enrichment":
        "299d9780-71e1-4845-b685-9a2f05c5994f",

    "m_inventory_reconciliation":
        "80f32658-8b80-4279-b844-002bb4195c9f",

    "m_daily_sales_summary":
        "c72222ce-55cd-4341-8982-2574a5b9aac7"
}

print("=" * 90)
print("HUMAN GATEWAY APPROVAL VERIFICATION")
print("=" * 90)

all_approved = True

for mapping_name, review_id in APPROVED_REVIEWS.items():

    review = gateway.get_review(review_id)

    if not review:
        raise ValueError(
            f"Review not found: {mapping_name}"
        )

    valid = (
        review.get("mapping_name") == mapping_name
        and review.get("status") == "PASS"
        and review.get("decision") == "APPROVE"
        and review.get("validations_passed") is True
        and review.get("can_execute") is True
    )

    if not valid:
        all_approved = False

    print({
        "mapping": mapping_name,
        "review_id": review_id,
        "status": review.get("status"),
        "decision": review.get("decision"),
        "validations_passed": review.get("validations_passed"),
        "can_execute": review.get("can_execute"),
        "verified": valid
    })

print("\n" + "=" * 90)

if all_approved:
    print("RESULT: ALL FOUR APPROVALS VERIFIED")
else:
    print("RESULT: ONE OR MORE APPROVALS NEED ATTENTION")

print("=" * 90)

In [0]:

import json

METADATA_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "parsed_metadata/idmc_parsed_objects.json"
)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

repository = metadata.get("repository", {})

print("=" * 90)
print("MAPPING SOURCE AND TARGET METADATA")
print("=" * 90)

for folder in repository.get("folders", []):

    for mapping in folder.get("mappings", []):

        mapping_name = mapping.get("name")

        print("\nMapping:", mapping_name)

        print("\nSources:")

        for source in mapping.get("source_instances", []):
            print({
                "source_name": source.get("name"),
                "source_table": (
                    source.get("source")
                    or source.get("object")
                )
            })

        print("\nTargets:")

        for target in mapping.get("target_instances", []):

            attributes = target.get("attributes", {})

            merge_keys = (
                attributes.get("merge_keys")
                or attributes.get("mergeKeys")
                or attributes.get("keyColumns")
                or attributes.get("primaryKey")
            )

            print({
                "target_name": target.get("name"),
                "target_table": (
                    target.get("target")
                    or target.get("object")
                ),
                "operation": target.get("operation"),
                "merge_keys": merge_keys,
                "merge_key_status": attributes.get(
                    "merge_key_status"
                )
            })

        print("-" * 90)


In [0]:

from pyspark.sql import functions as F
from functools import reduce

# ============================================================
# MERGE KEY PROFILING
# ============================================================

mapping_profiles = [
    {
        "mapping": "m_customer_standardize",
        "table": "idmc_poc.migration_test.src_customer_standardize",
        "keys": ["customer_id"],
        "check_uniqueness": True
    },
    {
        "mapping": "m_order_enrichment",
        "table": "idmc_poc.migration_test.src_orders",
        "keys": ["order_id"],
        "check_uniqueness": True
    },
    {
        "mapping": "m_inventory_reconciliation",
        "table": "idmc_poc.migration_test.src_inventory",
        "keys": ["product_id", "warehouse_id"],
        "check_uniqueness": True
    },
    {
        "mapping": "m_daily_sales_summary",
        "table": "idmc_poc.migration_test.src_orders",
        "keys": ["order_date"],
        "check_uniqueness": False
    }
]

profile_results = []

print("=" * 90)
print("MERGE KEY DATA PROFILING")
print("=" * 90)

for item in mapping_profiles:

    mapping_name = item["mapping"]
    table_name = item["table"]
    keys = item["keys"]

    print("\nMapping:", mapping_name)
    print("Source:", table_name)
    print("Candidate keys:", keys)

    try:
        if not spark.catalog.tableExists(table_name):
            print("RESULT: SOURCE TABLE NOT FOUND")

            profile_results.append({
                "mapping": mapping_name,
                "table": table_name,
                "status": "TABLE_NOT_FOUND"
            })
            continue

        df = spark.table(table_name)

        missing_columns = [
            key for key in keys
            if key not in df.columns
        ]

        if missing_columns:
            print("Missing columns:", missing_columns)

            profile_results.append({
                "mapping": mapping_name,
                "table": table_name,
                "status": "MISSING_KEY_COLUMNS",
                "missing_columns": missing_columns
            })
            continue

        total_rows = df.count()

        null_condition = reduce(
            lambda a, b: a | b,
            [F.col(key).isNull() for key in keys]
        )

        null_key_rows = df.filter(null_condition).count()

        print("Total rows:", total_rows)
        print("Rows with NULL key:", null_key_rows)

        result = {
            "mapping": mapping_name,
            "table": table_name,
            "keys": keys,
            "total_rows": total_rows,
            "null_key_rows": null_key_rows
        }

        if item["check_uniqueness"]:

            duplicate_groups = (
                df.groupBy(*keys)
                  .count()
                  .filter(F.col("count") > 1)
            )

            duplicate_key_groups = duplicate_groups.count()

            duplicate_rows = (
                duplicate_groups
                .agg(
                    F.sum("count").alias("duplicate_rows")
                )
                .first()["duplicate_rows"]
            )

            duplicate_rows = duplicate_rows or 0

            print("Duplicate key groups:", duplicate_key_groups)
            print("Rows belonging to duplicate groups:", duplicate_rows)

            result["duplicate_key_groups"] = duplicate_key_groups
            result["duplicate_rows"] = duplicate_rows

            result["status"] = (
                "PASS"
                if null_key_rows == 0
                and duplicate_key_groups == 0
                else "REVIEW_REQUIRED"
            )

        else:
            print(
                "Uniqueness check skipped: "
                "order_date is expected to repeat in raw orders."
            )

            result["status"] = "AGGREGATION_GRAIN_CHECK_REQUIRED"

        profile_results.append(result)

        print("Profiling status:", result["status"])

    except Exception as e:

        print("Profiling error:", str(e))

        profile_results.append({
            "mapping": mapping_name,
            "table": table_name,
            "status": "ERROR",
            "error": str(e)
        })

print("\n" + "=" * 90)
print("PROFILE SUMMARY")
print("=" * 90)

for result in profile_results:
    print(result)


In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# 1. Inspect NULL customer_id records
# ---------------------------------------------------------
print("\n" + "=" * 90)
print("1. CUSTOMER STANDARDIZATION - NULL CUSTOMER IDs")
print("=" * 90)

customer_df = spark.table(
    "idmc_poc.migration_test.src_customer_standardize"
)

customer_df.filter(
    F.col("customer_id").isNull()
).show(50, truncate=False)


# ---------------------------------------------------------
# 2. Inspect NULL order_id records
# ---------------------------------------------------------
print("\n" + "=" * 90)
print("2. ORDER ENRICHMENT - NULL ORDER IDs")
print("=" * 90)

orders_df = spark.table(
    "idmc_poc.migration_test.src_orders"
)

orders_df.filter(
    F.col("order_id").isNull()
).show(50, truncate=False)


# ---------------------------------------------------------
# 3. Check customer lookup join key
# ---------------------------------------------------------
print("\n" + "=" * 90)
print("3. CUSTOMER SILVER - CUSTOMER_ID JOIN KEY")
print("=" * 90)

customer_silver_table = "idmc_poc.migration_test.src_customer_silver"

if spark.catalog.tableExists(customer_silver_table):
    customer_silver_df = spark.table(customer_silver_table)

    print("Columns:", customer_silver_df.columns)
    print("Total rows:", customer_silver_df.count())

    if "customer_id" in customer_silver_df.columns:
        print(
            "NULL customer IDs:",
            customer_silver_df.filter(
                F.col("customer_id").isNull()
            ).count()
        )

        print(
            "Duplicate customer IDs:",
            customer_silver_df.groupBy("customer_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
else:
    print("Customer silver table not found.")


# ---------------------------------------------------------
# 4. Check inventory product lookup key
# ---------------------------------------------------------
print("\n" + "=" * 90)
print("4. PRODUCTS - PRODUCT_ID JOIN KEY")
print("=" * 90)

products_table = "idmc_poc.migration_test.src_products"

if spark.catalog.tableExists(products_table):
    products_df = spark.table(products_table)

    print("Columns:", products_df.columns)
    print("Total rows:", products_df.count())

    if "product_id" in products_df.columns:
        print(
            "NULL product IDs:",
            products_df.filter(
                F.col("product_id").isNull()
            ).count()
        )

        print(
            "Duplicate product IDs:",
            products_df.groupBy("product_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
else:
    print("Products table not found.")


# ---------------------------------------------------------
# 5. Inspect daily sales source columns
# ---------------------------------------------------------
print("\n" + "=" * 90)
print("5. DAILY SALES - SOURCE SCHEMA")
print("=" * 90)

print("Orders columns:", orders_df.columns)
orders_df.printSchema()


# ---------------------------------------------------------
# 6. Inspect generated PySpark code for all mappings
# ---------------------------------------------------------
print("\n" + "=" * 90)
print("6. GENERATED PYSPARK CODE")
print("=" * 90)

generated_dir = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark"
)

mapping_names = [
    "m_customer_standardize",
    "m_order_enrichment",
    "m_inventory_reconciliation",
    "m_daily_sales_summary"
]

for mapping_name in mapping_names:
    file_path = f"{generated_dir}/{mapping_name}.py"

    print("\n" + "-" * 90)
    print("MAPPING:", mapping_name)
    print("-" * 90)

    try:
        print(dbutils.fs.head(file_path, 20000))
    except Exception as e:
        print("Could not read generated file:", str(e))

In [0]:
# ---------------------------------------------------------
# 3. INVENTORY RECONCILIATION - CORRECTED
# ---------------------------------------------------------

inventory = spark.table(
    "idmc_poc.migration_test.src_inventory"
).alias("inv")

products = spark.table(
    "idmc_poc.migration_test.src_products"
).alias("prd")

inventory_final = (
    inventory
    .join(
        products,
        F.col("inv.product_id") == F.col("prd.product_id"),
        "left"
    )
    .select(
        F.col("inv.product_id").alias("product_id"),
        F.col("inv.warehouse_id").alias("warehouse_id"),
        F.col("inv.on_hand_qty").alias("on_hand_qty"),
        F.col("inv.reserved_qty").alias("reserved_qty"),
        F.col("inv.updated_ts").alias("updated_ts"),
        F.col("prd.product_name").alias("product_name"),
        F.col("prd.category").alias("category")
    )
    .withColumn(
        "available_qty",
        F.col("on_hand_qty") - F.col("reserved_qty")
    )
    .filter(F.col("available_qty") >= 0)
)

validate_key(
    "m_inventory_reconciliation",
    inventory_final,
    ["product_id", "warehouse_id"]
)

In [0]:
# ---------------------------------------------------------
# 4. DAILY SALES SUMMARY
# ---------------------------------------------------------

orders = spark.table(
    "idmc_poc.migration_test.src_orders"
)

daily_sales_final = (
    orders
    .withColumn(
        "order_date",
        F.to_date(F.col("order_ts"))
    )
    .filter(
        F.col("order_id").isNotNull() &
        (F.col("order_amount") >= 0)
    )
    .groupBy("order_date")
    .agg(
        F.count("order_id").alias("order_count"),
        F.sum("order_amount").alias("sales_total")
    )
)

validate_key(
    "m_daily_sales_summary",
    daily_sales_final,
    ["order_date"]
)

In [0]:
mappings = [
    "m_customer_standardize",
    "m_order_enrichment",
    "m_inventory_reconciliation",
    "m_daily_sales_summary"
]

latest_reviews = {}

print("=" * 90)
print("LATEST HUMAN GATEWAY REVIEWS")
print("=" * 90)

for mapping in mappings:
    review = gateway.get_review_by_mapping(
        mapping_name=mapping,
        status="REVIEW_REQUIRED"
    )

    if review:
        latest_reviews[mapping] = review

        print(f"\nMapping: {mapping}")
        print("Review ID:", review.get("review_id"))
        print("Status:", review.get("status"))
        print("Issues:", review.get("issues"))
        print("Validations passed:", review.get("validations_passed"))
    else:
        print(f"\n{mapping}: No pending review found")

In [0]:
# ============================================================
# HUMAN GATEWAY - APPROVE VALIDATED POC MERGE KEY REVIEWS
# ============================================================

review_approvals = {
    "m_customer_standardize": "f5f2499d-d3c9-4475-bed5-3cf7cf2085ff",
    "m_order_enrichment": "299d9780-71e1-4845-b685-9a2f05c5994f",
    "m_inventory_reconciliation": "80f32658-8b80-4279-b844-002bb4195c9f",
    "m_daily_sales_summary": "c72222ce-55cd-4341-8982-2574a5b9aac7"
}

reviewer = "POC_REVIEWER"

approval_results = []

for mapping_name, review_id in review_approvals.items():

    review = gateway.get_review(review_id)

    if review.get("mapping_name") != mapping_name:
        raise ValueError(
            f"Review mapping mismatch for {mapping_name}"
        )

    if review.get("status") != "REVIEW_REQUIRED":
        raise ValueError(
            f"{mapping_name} is not pending review: "
            f"{review.get('status')}"
        )

    if not review.get("validations_passed"):
        raise ValueError(
            f"Technical validations failed for {mapping_name}"
        )

    result = gateway.record_decision_by_id(
        review_id=review_id,
        decision="APPROVE",
        reviewer=reviewer,
        comment=(
            "Approved for controlled POC use. "
            "Final transformed DataFrame key profiling passed "
            "with zero NULL key rows and zero duplicate key groups. "
            "This approval is limited to the POC and does not "
            "constitute production business-key certification."
        )
    )

    approval_results.append({
        "mapping": mapping_name,
        "review_id": review_id,
        "result": result
    })

print("\n" + "=" * 90)
print("HUMAN GATEWAY APPROVAL RESULTS")
print("=" * 90)

for item in approval_results:
    print(item)

print("\nAll four review decisions processed.")

In [0]:
print("=" * 90)
print("APPROVAL VERIFICATION")
print("=" * 90)

for mapping_name, review_id in review_approvals.items():

    review = gateway.get_review(review_id)

    print({
        "mapping": mapping_name,
        "status": review.get("status"),
        "decision": review.get("decision"),
        "can_execute": review.get("can_execute")
    })